# Feature Engineering
- Create predictive features based on the raw data

In [2]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import PolynomialFeatures
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import train_test_split

In [3]:
clutch_df = pd.read_csv("../data/processed/clutch_shot_data.2023.csv")

print(clutch_df.shape)
clutch_df.head()

(1407, 26)


,GRID_TYPE,GAME_ID,GAME_EVENT_ID,PLAYER_ID,PLAYER_NAME,TEAM_ID,TEAM_NAME,PERIOD,MINUTES_REMAINING,SECONDS_REMAINING,...,SHOT_DISTANCE,LOC_X,LOC_Y,SHOT_ATTEMPTED_FLAG,SHOT_MADE_FLAG,GAME_DATE,HTM,VTM,TIME_REMAINING,CLUTCH
0,Shot Chart Detail,22300076,661,2544,LeBron James,1610612747,Los Angeles Lakers,4,1,11,...,2,-1,28,1,1,20231026,LAL,PHX,71,True
1,Shot Chart Detail,22300076,664,2544,LeBron James,1610612747,Los Angeles Lakers,4,0,41,...,1,9,16,1,1,20231026,LAL,PHX,41,True
2,Shot Chart Detail,22300100,704,2544,LeBron James,1610612747,Los Angeles Lakers,4,0,12,...,2,18,9,1,1,20231029,SAC,LAL,12,True
3,Shot Chart Detail,22300100,770,2544,LeBron James,1610612747,Los Angeles Lakers,5,0,50,...,3,32,13,1,1,20231029,SAC,LAL,50,True
4,Shot Chart Detail,22300100,776,2544,LeBron James,1610612747,Los Angeles Lakers,5,0,23,...,1,-6,13,1,1,20231029,SAC,LAL,23,True


## Feature Engineering

In [4]:
# PRESSURE_SCORE = Quantifies the pressure of a shot based on time remaining and shot distance
clutch_df['PRESSURE_SCORE'] = (
    (120 - clutch_df['TIME_REMAINING']) * 0.6 +
    (clutch_df['SHOT_DISTANCE']) * 0.4
)

# SHOT_DIFFICULTY = Combines shot distance and period
clutch_df['SHOT_DIFFICULTY'] = (
    clutch_df['SHOT_DISTANCE'] /
    (clutch_df['PERIOD'])
)

## Train/Test Split
Split **before** computing any player-level aggregates to avoid data leakage.

In [5]:
target = 'SHOT_MADE_FLAG'

train_df, test_df = train_test_split(
    clutch_df,
    test_size=0.2,
    random_state=42,
    stratify=clutch_df[target]
)

print(f"Train size: {len(train_df)}, Test size: {len(test_df)}")

Train size: 1125, Test size: 282


In [6]:
# PLAYER_CLUTCH_FG = Computed only on training data, then mapped to avoid leakage
train_player_fg = train_df.groupby('PLAYER_NAME')[target].mean()
global_mean_fg = train_df[target].mean()

train_df = train_df.copy()
test_df = test_df.copy()

train_df['PLAYER_CLUTCH_FG'] = train_df['PLAYER_NAME'].map(train_player_fg)
# For test, use training mean for unseen players
test_df['PLAYER_CLUTCH_FG'] = test_df['PLAYER_NAME'].map(train_player_fg).fillna(global_mean_fg)

## Drop Identifying Columns
Drop all identity/leakage columns **before** any encoding.

In [8]:
drop_cols = [
    'GRID_TYPE',
    'GAME_ID',
    'GAME_EVENT_ID',
    'GAME_DATE',
    'MATCHUP',
    'HTM',
    'VTM',
    'TEAM_NAME',
    'TEAM_ID',
    'PLAYER_ID',
    'PLAYER_NAME',   # dropped before encoding to prevent dummy leakage
    'SHOT_ATTEMPTED_FLAG',
    'EVENT_TYPE',
    'LOC_X',
    'LOC_Y'
]

def drop_id_cols(df):
    return df.drop(columns=[c for c in drop_cols if c in df.columns], errors='ignore')

train_df = drop_id_cols(train_df)
test_df  = drop_id_cols(test_df)

## Encoding

In [9]:
categorical_cols = [
    'ACTION_TYPE',
    'SHOT_ZONE_BASIC',
    'SHOT_ZONE_AREA',
    'SHOT_TYPE',
    'SHOT_ZONE_RANGE'
]
categorical_cols = [c for c in categorical_cols if c in train_df.columns]

# Fit dummies on train, align test to same columns
train_df = pd.get_dummies(train_df, columns=categorical_cols, drop_first=True)
test_df  = pd.get_dummies(test_df,  columns=categorical_cols, drop_first=True)

# Catch any remaining object columns
remaining_obj = train_df.select_dtypes(include=['object']).columns.tolist()
if remaining_obj:
    print("Warning: still have object columns after encoding:", remaining_obj)
    train_df = pd.get_dummies(train_df, columns=remaining_obj, drop_first=True)
    test_df  = pd.get_dummies(test_df,  columns=remaining_obj, drop_first=True)

# Align train and test to the same columns
train_df, test_df = train_df.align(test_df, join='left', axis=1, fill_value=0)

print("Train columns:", train_df.shape[1])
print("Test columns: ", test_df.shape[1])

Train columns: 68
Test columns:  68


In [10]:
# Convert bool columns to int
for df in [train_df, test_df]:
    bool_cols = df.select_dtypes(include=['bool']).columns
    df[bool_cols] = df[bool_cols].astype(int)

## Separate Features and Target

In [11]:
X_train = train_df.drop(columns=[target])
y_train = train_df[target]

X_test  = test_df.drop(columns=[target])
y_test  = test_df[target]

## Handle Inf / NaN

In [12]:
X_train = X_train.replace([np.inf, -np.inf], np.nan)
X_test  = X_test.replace([np.inf, -np.inf], np.nan)

# Compute medians on train only, apply to both
train_medians = X_train.median(numeric_only=True)
X_train = X_train.fillna(train_medians)
X_test  = X_test.fillna(train_medians)

## Polynomial Features

In [13]:
poly_features = ['SHOT_DISTANCE', 'TIME_REMAINING', 'PRESSURE_SCORE']
poly_features = [f for f in poly_features if f in X_train.columns]

poly = PolynomialFeatures(degree=2, include_bias=False)

poly_train = pd.DataFrame(
    poly.fit_transform(X_train[poly_features]),
    columns=poly.get_feature_names_out(poly_features),
    index=X_train.index
)
poly_test = pd.DataFrame(
    poly.transform(X_test[poly_features]),
    columns=poly.get_feature_names_out(poly_features),
    index=X_test.index
)

X_train = pd.concat([X_train.reset_index(drop=True), poly_train.reset_index(drop=True)], axis=1)
X_test  = pd.concat([X_test.reset_index(drop=True),  poly_test.reset_index(drop=True)],  axis=1)

## Scaling

In [14]:
numeric_cols = X_train.select_dtypes(include=['int64', 'float64']).columns

scaler = StandardScaler()
X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test[numeric_cols]  = scaler.transform(X_test[numeric_cols])

## Feature Selection

In [15]:
# Remove constant columns
constant_cols = X_train.columns[X_train.nunique() <= 1]
print("Constant features removed:", list(constant_cols))
X_train = X_train.drop(columns=constant_cols)
X_test  = X_test.drop(columns=constant_cols)

selector = SelectKBest(score_func=f_classif, k=min(25, X_train.shape[1]))
X_train_selected = selector.fit_transform(X_train, y_train)
X_test_selected  = selector.transform(X_test)

selected_features = X_train.columns[selector.get_support()]

print("\nSelected Features:")
print(selected_features.tolist())
print("\nFinal Feature Matrix Shape (train):", X_train_selected.shape)
print("Final Feature Matrix Shape (test): ", X_test_selected.shape)

Constant features removed: ['CLUTCH']

Selected Features:
['MINUTES_REMAINING', 'SHOT_DISTANCE', 'TIME_REMAINING', 'PRESSURE_SCORE', 'SHOT_DIFFICULTY', 'PLAYER_CLUTCH_FG', 'ACTION_TYPE_Cutting Dunk Shot', 'ACTION_TYPE_Driving Dunk Shot', 'ACTION_TYPE_Jump Shot', 'ACTION_TYPE_Pullup Jump shot', 'ACTION_TYPE_Running Dunk Shot', 'SHOT_ZONE_BASIC_Restricted Area', 'SHOT_ZONE_AREA_Center(C)', 'SHOT_ZONE_AREA_Left Side Center(LC)', 'SHOT_ZONE_AREA_Right Side Center(RC)', 'SHOT_TYPE_3PT Field Goal', 'SHOT_ZONE_RANGE_24+ ft.', 'SHOT_ZONE_RANGE_Less Than 8 ft.', 'SHOT_DISTANCE', 'TIME_REMAINING', 'PRESSURE_SCORE', 'SHOT_DISTANCE^2', 'SHOT_DISTANCE TIME_REMAINING', 'SHOT_DISTANCE PRESSURE_SCORE', 'PRESSURE_SCORE^2']

Final Feature Matrix Shape (train): (1125, 25)
Final Feature Matrix Shape (test):  (282, 25)


## Save Outputs

In [16]:
train_out = pd.DataFrame(X_train_selected, columns=selected_features)
train_out[target] = y_train.values
train_out.to_csv("../data/processed/engineered_clutch_train.2023.csv", index=False)

test_out = pd.DataFrame(X_test_selected, columns=selected_features)
test_out[target] = y_test.values
test_out.to_csv("../data/processed/engineered_clutch_test.2023.csv", index=False)

print("Saved train and test CSVs.")

Saved train and test CSVs.
